# Gas Turbine · добавляют ли тройки признаков точность на NOx и CO?
На девяти общих признаках сравниваем три заранее фиксированные конфигурации второй ступени.

| Модель | Архитектура | Параметры |
|---|---|---:|
| MLP | 2 скрытых ReLU-слоя ширины 41 | 2 174 |
| Парная MGS-KAN | 16 ядер на признак и 8 на пару | 2 170 |
| Гиперграфовая MGS-KAN | 9 ядер на признак, 5 на пару и 1 на каждую из 84 троек | 2 173 |

В MLflow должны быть готовы 12 чекпоинтов MLP и парной MGS-KAN из эксперимента `gas_turbine_heldout` (валидационные запуски второй волны). При неполной истории ноутбук укажет проблемную серию до обучения. Для гиперграфа запланированы шесть запусков, по три пары seed для каждой цели. Выбери GPU и выполни **Run All**. Повторный запуск подхватит готовые чекпоинты.

Обучение на 2011–2012 годах, Adam, 100 эпох, шаг 0,001, батч 64. По данным 2013 года выбираются лучшая эпоха и рейтинг троек. Данные 2014–2015 годов используются только для итоговой оценки за весь период.

In [ ]:
PROJECT_DIR = None
DEVICE = "cuda:1"
GPU_BUDGET_MIB = 4096
VERSION = "gas-hypergraph-v1"

In [ ]:
import sys
from pathlib import Path

if PROJECT_DIR is None:
    cwd = Path.cwd().resolve()
    checkout = next(
        (folder for folder in (cwd, *cwd.parents) if (folder / "src/experiments/runner.py").is_file()),
        None,
    )
else:
    checkout = Path(PROJECT_DIR).expanduser()
    if not checkout.is_absolute() or not (checkout / "src/experiments/runner.py").is_file():
        raise ValueError("Укажи в PROJECT_DIR абсолютный путь к корню mgs-kan на сервере.")
    checkout = checkout.resolve()

if checkout is not None and sys.path[0:1] != [str(checkout)]:
    sys.path.insert(0, str(checkout))

try:
    from src.paths import DATA_DIR, ENV_FILE, PROJECT_ROOT
except ModuleNotFoundError as error:
    raise ImportError("Укажи PROJECT_DIR и выполни ячейки заново.") from error

if checkout is not None and PROJECT_ROOT != checkout:
    raise RuntimeError("Ядро уже загрузило src из другой папки. Перезапусти ядро.")
if not (PROJECT_ROOT / "notebooks/baselines/run_series.ipynb").is_file():
    raise RuntimeError("Укажи PROJECT_DIR с рабочей копией mgs-kan и перезапусти ядро.")
print(f"Проект: {PROJECT_ROOT}\nНастройки: {ENV_FILE}\nДанные: {DATA_DIR}")

In [ ]:
import json
import math
from itertools import product

import pandas as pd
import torch
from IPython.display import display

from src.config import ModelSettings, MonitorSettings, ResourceSettings, TrackingSettings
from src.data import load_dataset
from src.experiments.search import SearchStage, ValidationGrid
from src.experiments.triples import rank_validation_triples
from src.models import make_model, parameter_count
from src.tracking import MlflowTracker
from src.training import Trainer

targets = ("gas_turbine_nox", "gas_turbine_co")
seeds = ((11, 101), (22, 202), (33, 303))
hypergraph = "hypergraph_mgs_kan"
counts = {"mlp": 2174, "mgs_kan": 2170, hypergraph: 2173}
survey = ValidationGrid.preset("benchmark_survey", datasets=targets)
baseline = survey.wave(2).stages[0].model_copy(update={"models": ("mlp", "mgs_kan"), "history_only": True})
third_order = SearchStage(name="third-order", models=(hypergraph,),
                          settings=ModelSettings(single_kernels=9, pair_kernels=5))
grid = ValidationGrid.model_validate({**survey.model_dump(), "stages": (baseline, third_order), "seeds": seeds})
tracking = TrackingSettings(_env_file=ENV_FILE)
monitoring = MonitorSettings(_env_file=ENV_FILE)
resources = ResourceSettings(_env_file=ENV_FILE, max_parallel_trials=1, cpu_threads=1,
                             gpu_budget_mib=GPU_BUDGET_MIB)
data = {target: load_dataset(target, root=DATA_DIR, fold=0) for target in targets}
settings_by_model = {}
for target, _, _, stage, settings in grid.plan():
    opts = settings.model_copy(update={"features": data[target].n_features, "init_seed": 11})
    for name in stage.models:
        if parameter_count(make_model(name, opts)) != counts[name]:
            raise ValueError(f"Unexpected parameter count for {name}")
        settings_by_model[target, name] = opts
output = PROJECT_ROOT / "results" / "gas-turbine-hypergraph" / VERSION
print(f"{len(targets)} цели · {len(seeds)} пары seed · {DEVICE}")

In [ ]:
runs = grid.run(version=VERSION, device=DEVICE, root=DATA_DIR, resources=resources,
                tracking=tracking, monitoring=monitoring)
expected = {(t, m, i, s) for t, m, (i, s) in product(targets, counts, seeds)}
keys = ["dataset", "model", "init_seed", "shuffle_seed"]
if (len(runs) != len(expected) or runs.run_id.nunique() != len(expected)
        or set(runs[keys].itertuples(index=False, name=None)) != expected):
    raise ValueError("Expected 18 distinct paired runs")
print(f"Взято из MLflow: {runs.reused.sum()}. Новых обучений: {(~runs.reused).sum()}.")

## Какие тройки использует модель?
У каждой тройки есть свой аддитивный член. Для каждого сохранённого гиперграфа отключаем один член за раз при фиксированных остальных весах. Оценка — `delta_val_rmse = RMSE без тройки − RMSE полной модели` на 2013 году. Положительное значение означает, что отключение ухудшило прогноз. В топ-5 входят пять максимальных значений для каждой пары seed и цели, при равенстве — по имени тройки.

Дополнительный показатель `mixed_strength` — корень из средней квадратичной смешанной разности третьего порядка, делённой под корнем на 8. Для каждой строки используем вторую строку из фиксированной перестановки валидации и все восемь комбинаций трёх координат. Показатель гасит аддитивные и чисто парные эффекты. Гибридные сочетания коррелирующих датчиков могут быть редкими в эксплуатации. Рейтинг характеризует выученную модель.

In [ ]:
torch.set_num_threads(resources.cpu_threads)  # Восстановление весов и анализ выполняются на CPU.
tracker = MlflowTracker(tracking)
verified = []
rankings = []
for row in runs.itertuples(index=False):
    dataset = data[row.dataset]
    params = tracker.client.get_run(row.run_id).data.params
    if (params["dataset_sha256"] != dataset.source_sha256 or params["target_column"] != dataset.target_column
            or json.loads(params["feature_columns"]) != list(dataset.columns)):
        raise ValueError(f"Dataset mismatch for {row.run_id}")
    opts = settings_by_model[row.dataset, row.model].model_copy(update={"init_seed": row.init_seed})
    trainer = Trainer(make_model(row.model, opts), tracker=tracker)
    if trainer.load_checkpoint(run_id=row.run_id) != row.best_epoch:
        raise ValueError(f"Unexpected checkpoint epoch for {row.run_id}")
    x, y = dataset.val.tensors
    pred = torch.cat([trainer.predict(batch) for batch in x.split(grid.training.batch_size)])
    residual = dataset.inverse_target(pred.double()) - dataset.inverse_target(y.double())
    if not math.isclose(residual.square().mean().sqrt().item(), row.val_rmse, rel_tol=1e-4, abs_tol=1e-4):
        raise ValueError(f"Validation RMSE mismatch for {row.run_id}")
    verified.append((row, trainer))
    if row.model == hypergraph:
        scores = rank_validation_triples(trainer.model, dataset)
        rankings.append(scores.assign(dataset=row.dataset, init_seed=row.init_seed,
                                      shuffle_seed=row.shuffle_seed, run_id=row.run_id))
ranks = pd.concat(rankings, ignore_index=True)
ranks["in_top5"] = ranks["rank"] <= 5
stability = ranks.groupby(["dataset", "triple"], as_index=False).agg(
    top5_count=("in_top5", "sum"), mean_rank=("rank", "mean"),
    mean_delta_val_rmse=("delta_val_rmse", "mean"), mean_mixed_strength=("mixed_strength", "mean"),
).sort_values(["dataset", "top5_count", "mean_rank", "triple"], ascending=[True, False, True, True])
for target in targets:
    repeated = stability.loc[(stability.dataset == target) & (stability.top5_count >= 2)]
    print(f"{target}: в топ-5 при всех трёх seed — {(repeated.top5_count == 3).sum()} троек.")
    display(repeated.round(5))

## RMSE на тесте 2014–2015 годов
Оцениваем полные модели со всеми их членами. Для каждой цели сравниваем одни и те же три пары seed. Отрицательная разность означает меньшую тестовую RMSE у гиперграфа. В таблицах сохраняются все 84 тройки каждого гиперграфа, включая случаи, когда ни одна не повторилась в топ-5 трижды.

In [ ]:
rows = []
for row, trainer in verified:
    dataset = data[row.dataset]
    rows.append(dict(row._asdict(), test_rmse=trainer.evaluate(dataset), n_test=len(dataset.test)))
test_runs = pd.DataFrame(rows)
paired = test_runs.pivot(index=["dataset", "init_seed", "shuffle_seed"], columns="model", values="test_rmse")
paired["delta_vs_pair"] = paired[hypergraph] - paired["mgs_kan"]
paired["delta_vs_mlp"] = paired[hypergraph] - paired["mlp"]
summary = paired.groupby("dataset").mean()
display(summary.round(4))
display(paired.round(4))
output.mkdir(parents=True, exist_ok=True)
test_runs.to_csv(output / "runs.csv", index=False)
paired.to_csv(output / "paired.csv")
summary.to_csv(output / "summary.csv")
ranks.to_csv(output / "validation-triples.csv", index=False)
stability.to_csv(output / "top5-stability.csv", index=False)
print(f"Результаты: {output}")